# Thu thập, lọc và gán nhãn bệnh nhân
Mục tiêu chính: Dựa trên dữ liệu 12h đầu nhập ICU của 1 bệnh nhân AMI -> Dự đoán bệnh nhân đó trong 7 ngày tới có bị bệnh thận không?
1. Thu thập bệnh nhân trên MIMIC - database chứa thông tin các bệnh nhân
2. Giữ lại những bệnh nhân AMI, đã nằm ICU ít nhất 12 tiếng, và trong khoảng thời gian đấy không bị bệnh thận
3. Gán nhãn bệnh nhân có bị bệnh/không bị bệnh trong 7 ngày sau dựa trên bộ tiêu chí có sẵn.

In [1]:
import pandas as pd
import numpy as np
import glob
import re
import os
os.chdir('/media/data3/biodataset/MIMIC_IV/MIMIC-IV-v3.0/physionet.org/files/mimiciv/3.0/')

### 1. Thu thập bệnh nhân bị AMI
- Lọc bệnh nhân AMI theo mã ICD9 và ICD10
- Loại những bệnh nhân bị renal failure theo mã ICD9, ICD10
- Với từng hadm_id, chỉ giữ ICU stays đầu tiên (Tránh leakage)
- Lọc bệnh nhân <= 18t, hoặc >= 90t
- Lọc những bệnh nhân có ICU stays < 12 tiếng

In [2]:
pd.read_csv('hosp/diagnoses_icd.csv')

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9
...,...,...,...,...,...
6364515,19999987,23865745,7,41401,9
6364516,19999987,23865745,8,78039,9
6364517,19999987,23865745,9,0413,9
6364518,19999987,23865745,10,36846,9


In [ ]:
import pandas as pd

icu_cols = ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'los', 'first_careunit', 'last_careunit']
diag_cols = ['subject_id', 'hadm_id', 'icd_code', 'icd_version', 'seq_num']
pat_cols = ['subject_id', 'gender', 'anchor_age', 'anchor_year']

# Đọc dữ liệu và lấy các bệnh nhân bị AMI
df_icu = pd.read_csv('icu/icustays.csv', usecols=icu_cols)
df_diag = pd.read_csv('hosp/diagnoses_icd.csv', usecols=diag_cols)
df_pat = pd.read_csv('hosp/patients.csv', usecols=pat_cols)

df_diag['icd_code'] = df_diag['icd_code'].astype(str)

is_ami = (
    ((df_diag['icd_version'] == 9) & (df_diag['icd_code'].str.startswith('410'))) | 
    ((df_diag['icd_version'] == 10) & (df_diag['icd_code'].str.startswith('I21')))
)
df_ami_diag = df_diag[is_ami & (df_diag['seq_num'] == 1)][['subject_id', 'hadm_id']].drop_duplicates()

df_icu_pat = pd.merge(df_icu, df_pat, on='subject_id', how='inner')
df_icu_pat['intime'] = pd.to_datetime(df_icu_pat['intime'])
df_icu_pat = df_icu_pat.sort_values(by=['subject_id', 'hadm_id', 'intime'])
df_icu_pat = df_icu_pat.drop_duplicates(subset=['hadm_id'], keep='first')

# Loại các bệnh nhân không thỏa mãn độ tuổi và có tiền sử suy thận nặng rồi
df_icu_pat['admission_age'] = df_icu_pat['anchor_age'] + (df_icu_pat['intime'].dt.year - df_icu_pat['anchor_year']) # anchor_age, anchor_year là tuổi của bn tại năm mốc
mask_age = (df_icu_pat['admission_age'] >= 18) & (df_icu_pat['admission_age'] <= 90)
mask_los = (df_icu_pat['los'] >= 0.5)
df_filtered_icu = df_icu_pat[mask_age & mask_los]

final_cohort = pd.merge(
    df_filtered_icu,
    df_ami_diag,
    on=['subject_id', 'hadm_id'],
    how='inner'
)

excl_icd9_list = [
    '40301', '40311', '40391', '40402', '40403', '40412', '40413', '40492', '40493',
    '5855', '5856', 'V451', 'V4511', 'V4512', 'V560', 'V561', 'V562', 'V5631', 'V5632', 'V568'
]

excl_icd10_list = [
    'I120', 'I1311', 'I132', 'N185', 'N186', 
    'Z49', 'Z490', 'Z4901', 'Z4902', 'Z493', 'Z4931', 'Z4932', 'Z992'
]

renal_mask = (
    ((df_diag['icd_version'] == 9) & (df_diag['icd_code'].isin(excl_icd9_list))) | 
    ((df_diag['icd_version'] == 10) & (df_diag['icd_code'].isin(excl_icd10_list)))
)

renal_hadm_ids = df_diag[renal_mask]['hadm_id'].unique()

print(f"Số lượng đợt nhập viện (hadm_id) có tiền sử bệnh thận: {len(renal_hadm_ids)}")

# Loại bỏ các hadm_id này khỏi cohort
final_cohort = final_cohort[~final_cohort['hadm_id'].isin(renal_hadm_ids)]
final_cohort = final_cohort.sort_values(by=['subject_id', 'stay_id'])
final_cohort = final_cohort.drop_duplicates(subset=['subject_id'], keep='first')

print("-" * 30)
print(f"TỔNG SỐ BẢN GHI SAU KHI LOẠI TIỀN SỬ THẬN (THEO HADM_ID): {len(final_cohort)}")
print("-" * 30)
print(final_cohort[['subject_id', 'hadm_id', 'stay_id', 'admission_age', 'los']].head())

Đang đọc dữ liệu...
Số lượng đợt nhập viện (hadm_id) có tiền sử bệnh thận: 19720
------------------------------
TỔNG SỐ BẢN GHI SAU KHI LOẠI TIỀN SỬ THẬN (THEO HADM_ID): 3389
------------------------------
   subject_id   hadm_id   stay_id  admission_age       los
0    10002155  23822395  33685454             81  6.178912
1    10002495  24982426  36753294             81  5.087512
2    10002667  23197839  31573075             58  1.951921
3    10005817  20626031  32604416             66  2.359097
4    10007058  22954658  32506122             48  2.022963


In [4]:
final_cohort

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,admission_age
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,80,2128,81
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,2141,81
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,2187,58
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,2132,66
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,2167,48
...,...,...,...,...,...,...,...,...,...,...,...,...
3654,19990821,27777812,38906628,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2143-03-06 16:21:17,2143-03-07 06:40:14,0.596493,M,70,2143,70
3655,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,80,2126,84
3656,19995780,21942461,36805359,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2125-10-20 11:32:13,2125-10-23 20:44:30,3.383530,M,84,2125,84
3658,19996783,21880161,31700331,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2188-05-10 02:20:42,2188-05-10 22:30:28,0.840116,M,89,2188,89


- Loại những bệnh nhân bị bệnh thận trước 12h nhập ICU theo tiêu chuẩn KDIGO

In [5]:
final_cohort['prediction_start_time'] = final_cohort['intime'] + pd.Timedelta(hours=12)

print("Đang đọc dữ liệu KDIGO AKI...")
df_kdigo = pd.read_csv('/media/data3/users/tpp/mimic-derived/organfailure/kdigo_stages.csv') # csv chứa sẵn thông tin tại từng mốc thời gian bệnh nhân có bị bệnh không
df_kdigo['charttime'] = pd.to_datetime(df_kdigo['charttime'])

df_aki_check = df_kdigo[['stay_id', 'charttime', 'aki_stage_smoothed']]

merged_check = pd.merge(
    final_cohort[['stay_id', 'prediction_start_time']], 
    df_aki_check, 
    on='stay_id', 
    how='inner'
)

pre_observation_aki_mask = (
    (merged_check['charttime'] <= merged_check['prediction_start_time']) & 
    (merged_check['aki_stage_smoothed'] > 0)
)

stay_ids_with_pre_aki = merged_check.loc[pre_observation_aki_mask, 'stay_id'].unique()

print(f"Số lượng bệnh nhân bị loại vì có AKI trước mốc 12h: {len(stay_ids_with_pre_aki)}")

final_cohort_clean = final_cohort[~final_cohort['stay_id'].isin(stay_ids_with_pre_aki)].copy()

Đang đọc dữ liệu KDIGO AKI...
Số lượng bệnh nhân bị loại vì có AKI trước mốc 12h: 1073


In [6]:
final_cohort_clean

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,admission_age,prediction_start_time
0,10002155,23822395,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,F,80,2128,81,2129-08-05 00:45:00
1,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,2141,81,2141-05-23 08:18:01
2,10002667,23197839,31573075,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2187-02-23 16:02:25,2187-02-25 14:53:11,1.951921,F,58,2187,58,2187-02-24 04:02:25
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,M,66,2132,66,2132-12-15 21:29:01
4,10007058,22954658,32506122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2167-11-07 20:22:00,2167-11-09 20:55:04,2.022963,M,48,2167,48,2167-11-08 08:22:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3649,19981210,26790133,38232926,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2148-07-06 21:47:34,2148-07-10 16:54:16,3.796319,M,73,2143,78,2148-07-07 09:47:34
3651,19986880,22950392,37092788,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2185-07-08 01:08:06,2185-07-14 17:19:57,6.674896,F,78,2185,78,2185-07-08 13:08:06
3652,19989437,25692702,34737716,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2181-09-14 08:47:07,2181-09-18 22:26:10,4.568785,M,38,2181,38,2181-09-14 20:47:07
3655,19991743,20346977,36005469,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2130-10-05 14:59:00,2130-10-10 21:52:49,5.287373,F,80,2126,84,2130-10-06 02:59:00


### 2. Gán nhãn suy thận (AKI) cho những bệnh nhân trên theo tiêu chuẩn 
- Xác định khoảng thời gian (12h ICU -> 7 ngày tới)
- Dựa trên file label có sẵn, nếu trong khoảng thời gian trên giai đoạn suy thận >= 1 -> Label cuối cho bệnh nhân = 1 (Có bị bệnh)

In [9]:
df_labeling = final_cohort_clean[['subject_id', 'stay_id', 'intime']].copy()

df_labeling['prediction_start_time'] = df_labeling['intime'] + pd.Timedelta(hours=12)
df_labeling['prediction_end_time'] = df_labeling['prediction_start_time'] + pd.Timedelta(days=7)

df_kdigo_subset = df_kdigo[df_kdigo['stay_id'].isin(df_labeling['stay_id'])].copy()

stay_ids_with_kdigo = df_kdigo_subset['stay_id'].unique()
df_labeling = df_labeling[df_labeling['stay_id'].isin(stay_ids_with_kdigo)]
final_cohort_clean = final_cohort_clean[final_cohort_clean['stay_id'].isin(stay_ids_with_kdigo)]

print(f"Số bệnh nhân sau khi lọc (có dữ liệu KDIGO): {len(df_labeling)}")

merged_label = pd.merge(
    df_labeling,
    df_kdigo_subset[['stay_id', 'charttime', 'aki_stage_smoothed']],
    on='stay_id',
    how='inner'  
)

mask_window = (
    (merged_label['charttime'] >= merged_label['prediction_start_time']) & 
    (merged_label['charttime'] <= merged_label['prediction_end_time'])
)

events_in_window = merged_label[mask_window]

max_aki_in_window = events_in_window.groupby('stay_id')['aki_stage_smoothed'].max().reset_index()
max_aki_in_window.rename(columns={'aki_stage_smoothed': 'max_aki_stage_7d'}, inplace=True)

final_cohort_labeled = pd.merge(
    final_cohort_clean,
    max_aki_in_window,
    on='stay_id',
    how='left'
)

final_cohort_labeled = pd.merge(
    final_cohort_labeled,
    df_labeling[['stay_id', 'prediction_end_time']],
    on='stay_id',
    how='left'
)

# Những người KHÔNG có AKI trong window (nhưng có dữ liệu KDIGO) → Gán 0
final_cohort_labeled['max_aki_stage_7d'] = final_cohort_labeled['max_aki_stage_7d'].fillna(0)
final_cohort_labeled['aki_label'] = (final_cohort_labeled['max_aki_stage_7d'] >= 1).astype(int)

print("-" * 30)
print("THỐNG KÊ NHÃN (LABEL DISTRIBUTION):")
print(final_cohort_labeled['aki_label'].value_counts())
print(f"Tỷ lệ dương tính (AKI): {final_cohort_labeled['aki_label'].mean():.2%}")
print("-" * 30)

# print("-" * 30)
# print("PHÂN TÍCH CHI TIẾT CÁC CA AKI (LABEL = 1)")
# print("-" * 30)

# aki_stage_counts = final_cohort_labeled[final_cohort_labeled['aki_label'] == 1]['max_aki_stage_7d'].value_counts().sort_index()
# print("Phân bố theo Stage (1, 2, 3):")
# print(aki_stage_counts)
# print(f"Tỷ lệ Stage 1 (Nhẹ) trong tổng số ca AKI: {aki_stage_counts.get(1, 0) / 1233:.2%}")

# print("-" * 30)

Số bệnh nhân sau khi lọc (có dữ liệu KDIGO): 2316
------------------------------
THỐNG KÊ NHÃN (LABEL DISTRIBUTION):
aki_label
1    1478
0     838
Name: count, dtype: int64
Tỷ lệ dương tính (AKI): 63.82%
------------------------------


In [8]:
final_cohort_labeled.drop(['anchor_age', 'anchor_year', 'max_aki_stage_7d'], axis=1, inplace=True)
final_cohort_labeled.to_csv('/media/data3/users/tpp/MIMICIV_AKI_AMI/Thai/AKI_patients.csv')